In [1]:
import pandas as pd
import sqlite3

connection = sqlite3.connect("RecruitSmart.db")

In [3]:
query = """

SELECT

ps.person_id,

COUNT(ps.skill) AS total_skills,

CASE

WHEN COUNT(ps.skill) <= 10 THEN 0.2

WHEN COUNT(ps.skill) <= 30 THEN 0.5

ELSE 0.9

END AS match_relevance_score

FROM PersonSkills ps

GROUP BY ps.person_id

ORDER BY total_skills DESC;

"""

target_scores = pd.read_sql(query, connection)

target_scores.head()

,person_id,total_skills,match_relevance_score
0,54188,441,0.9
1,35877,441,0.9
2,17566,441,0.9
3,49359,433,0.9
4,31048,433,0.9


In [5]:
query = """

SELECT

a.application_id,

j.job_id,

j.title,

j.required_experience,

p.person_id,

p.name

FROM Applications a

JOIN Jobs j

ON a.job_id = j.job_id

JOIN Candidates p

ON a.person_id = p.person_id;

"""

feature_matrix = pd.read_sql(query, connection)

feature_matrix.head()

,application_id,job_id,title,required_experience,person_id,name
0,1,15796,Beauty & Fragrance consultants needed,None,20608,MySQL Database Administrator
1,2,861,Retail Territory Merchandiser,None,37600,MSSQL/MYSQL/PostgreSQL Database Administrator
2,3,5391,Inside Sales Rep,Associate,51921,Shmitt Technologies
3,4,11965,Cities Project Manager,None,28530,VP of Information Technology
4,5,11285,Digital Procurement Assistant,Entry level,35955,Python Developer


In [7]:
feature_matrix = feature_matrix.merge(
    target_scores,
    on="person_id",
    how="left"
)

feature_matrix.head()

,application_id,job_id,title,required_experience,person_id,name,total_skills,match_relevance_score
0,1,15796,Beauty & Fragrance consultants needed,None,20608,MySQL Database Administrator,19,0.5
1,2,861,Retail Territory Merchandiser,None,37600,MSSQL/MYSQL/PostgreSQL Database Administrator,113,0.9
2,3,5391,Inside Sales Rep,Associate,51921,Shmitt Technologies,66,0.9
3,4,11965,Cities Project Manager,None,28530,VP of Information Technology,26,0.5
4,5,11285,Digital Procurement Assistant,Entry level,35955,Python Developer,58,0.9


In [9]:
feature_matrix.to_csv(
    "flattened_feature_matrix.csv",
    index=False
)

print("Feature matrix saved successfully!")

Feature matrix saved successfully!


In [11]:
feature_matrix["match_relevance_score"].value_counts()

match_relevance_score
0.9    513
0.5    376
0.2    111
Name: count, dtype: int64

In [13]:
feature_matrix.groupby("job_id")["match_relevance_score"].mean().head(20)

job_id
10     0.7
56     0.9
78     0.5
118    0.2
127    0.5
162    0.5
190    0.5
191    0.5
198    0.5
203    0.9
207    0.9
236    0.2
302    0.9
303    0.9
385    0.9
392    0.5
405    0.9
413    0.5
418    0.9
420    0.2
Name: match_relevance_score, dtype: float64

In [15]:
connection.close()

print("Database Closed Successfully!")

Database Closed Successfully!
